# Stage 3 & 4 — Adaptive Method Ablation

Sweep over memory budgets, recent-window sizes, and compress dtypes.
Analyze the quality–efficiency tradeoff.

# Stage 3 — Adaptive KV Cache Validation

This notebook validates all Stage 3 components using GPT-2 before running official experiments on Mistral-7B.

| Component | File | Role |
|-----------|------|------|
| `ImportanceScorer` | `src/adaptive/importance_scorer.py` | Accumulates per-token attention weights across all layers |
| `CacheCompressor` | `src/adaptive/compressor.py` | Per-token INT8 quantization of key/value tensors |
| `EvictionPolicy` | `src/adaptive/eviction_policy.py` | Selects lowest-scored tokens to evict when memory budget exceeded |
| `AdaptiveCacheManager` | `src/adaptive/cache_manager.py` | Orchestrates all three components at each decoding step |
| `run_adaptive()` | `src/models/patched_attention.py` | Decoding loop using `output_attentions=True` |

**Token state machine (one direction only):**
```
full_precision → compressed → evicted
```
The recent window (last 256 tokens) is always kept in full precision and protected from eviction.

## 1. Setup

In [ ]:
# Clone repo and install dependencies (run once per Colab session)
!git clone https://github.com/yanghao13111/adaptive-kv-cache.git 2>/dev/null || echo "Repo already cloned"
%cd adaptive-kv-cache
!pip install transformers accelerate datasets -q

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/adaptive-kv-cache')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Validate `ImportanceScorer`

`ImportanceScorer` accumulates attention weights that each cached token has received as a *key*,
averaged across heads, summed across layers.

**Why this works:** Tokens that are frequently attended to carry important information the model
actively uses. Tokens with near-zero accumulated attention are safe to compress or evict.

After one update with 2 fake layers:
- Layer 1 attention to 4 keys: `[0.6, 0.2, 0.1, 0.1]`
- Layer 2 attention to 4 keys: `[0.5, 0.3, 0.1, 0.1]`
- Expected score: element-wise sum = `[1.1, 0.5, 0.2, 0.2]`

In [ ]:
from src.adaptive.importance_scorer import ImportanceScorer

scorer = ImportanceScorer(num_layers=2)
fake_attn = (
    torch.tensor([[[[0.6, 0.2, 0.1, 0.1]]]]),  # layer 1: (1, 1, 1, 4)
    torch.tensor([[[[0.5, 0.3, 0.1, 0.1]]]]),  # layer 2
)
scorer.update(fake_attn)
scores = scorer.score(4)
print('ImportanceScorer scores:', scores.tolist())
# Expected: [1.1, 0.5, 0.2, 0.2]
assert abs(scores[0].item() - 1.1) < 1e-4, f'Expected 1.1, got {scores[0].item()}'
print('ImportanceScorer OK ✓')

## 3. Validate `CacheCompressor`

`CacheCompressor` uses per-token INT8 quantization: each token's key/value vector gets its own
scale and zero_point, computed from that vector's min and max.

**Key implementation detail:** The zero_point must NOT be clamped to the quantization range.
If clamped, the affine mapping breaks for tensors where the natural zero_point falls outside [-128, 127].

Expected: round-trip reconstruction error < 0.01 on random FP16 tensors.

In [ ]:
from src.adaptive.compressor import CacheCompressor

compressor = CacheCompressor(dtype='int8')
tensor = torch.randn(1, 2, 3, 4, dtype=torch.float16)
quantized, metadata = compressor.compress(tensor)
restored = compressor.decompress(quantized, metadata)
max_err = (tensor.float() - restored.float()).abs().max().item()
print(f'CacheCompressor max error: {max_err:.5f}')
# Expected: < 0.01
assert max_err < 0.01, f'Error too large: {max_err}'
print('CacheCompressor OK ✓')

## 4. Validate `EvictionPolicy`

`EvictionPolicy` selects the lowest-scored tokens for eviction, with two constraints:
1. Only tokens outside the `recent_window` are candidates
2. Returns exactly `n_evict` indices, sorted ascending

Test: scores `[0.9, 0.1, 0.3, 0.8, 0.2]`, recent_window=2, evict 2.
- Tokens 3 and 4 (indices 3, 4) are in the recent window — protected
- Among tokens 0–2: scores are `[0.9, 0.1, 0.3]`
- Lowest 2: indices **1** (0.1) and **2** (0.3)

In [ ]:
from src.adaptive.eviction_policy import EvictionPolicy

policy = EvictionPolicy(memory_budget_gb=0.001, recent_window=2)
scores = torch.tensor([0.9, 0.1, 0.3, 0.8, 0.2])
indices = policy.select_eviction_indices(scores, n_evict=2)
print('Evict indices:', indices.tolist())
# Expected: [1, 2]
assert indices.tolist() == [1, 2], f'Expected [1, 2], got {indices.tolist()}'
print('EvictionPolicy OK ✓')

## 5. Validate Full Pipeline — `run_adaptive()` with GPT-2

End-to-end test of the complete adaptive pipeline:
`run_adaptive()` → `AdaptiveCacheManager.step()` → scorer + compressor + eviction at every token.

We use a tight memory budget (0.5 GB) to force eviction to trigger during generation.
Expected output: coherent text, latency metrics, and peak memory near the budget.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.adaptive.cache_manager import AdaptiveCacheManager
from src.models.patched_attention import run_adaptive

model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to(DEVICE)
model.eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id

cache_manager = AdaptiveCacheManager(
    memory_budget_gb=0.5,
    recent_window=8,
    compress_dtype='int8',
    num_layers=12,
)

result = run_adaptive(
    model, tokenizer,
    'Once upon a time',
    cache_manager,
    max_new_tokens=20,
    device=DEVICE,
)
print('Generated:', result['generated_text'])
print(f'Latency: {result["latency_ms_per_token"]:.2f} ms/token')
print(f'Memory:  {result["peak_memory_gb"]:.3f} GB')
assert len(result['generated_text']) > 0, 'No output generated'
print('Full pipeline OK ✓')

## Stage 3 Validation Summary

| Component | Test | Status |
|-----------|------|--------|
| `ImportanceScorer` | Scores `[1.1, 0.5, 0.2, 0.2]` from 2-layer fake attention | ✓ Validated |
| `CacheCompressor` | INT8 round-trip max error < 0.01 | ✓ Validated |
| `EvictionPolicy` | Evicts indices `[1, 2]` from `[0.9, 0.1, 0.3, 0.8, 0.2]` with window=2 | ✓ Validated |
| `run_adaptive()` | End-to-end GPT-2 generation with eviction + compression | ✓ Validated |

All Stage 3 components validated. Next: **Stage 4** — official experiments on Mistral-7B with WikiText-103 and PG-19.